In [9]:
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright
import pandas as pd
import numpy as np
import os
import threading

def fighters_subpage_url_scraper():
    base_fighter_url = 'http://ufcstats.com/statistics/fighters?char=' # + str(letter)
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    subpage_url_suffice = '&page=all'

    alphabetized_urls = []
    for letter in alphabet:
        alphabetized_urls.append(base_fighter_url + str(letter) + subpage_url_suffice)

    return alphabetized_urls

def single_fighter_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    fighter_soup = soup.find('section', class_ = 'b-statistics__section_details')

    #For record
    heading_soup = soup.find('h2')
    record_unclean = heading_soup.find('span', class_ = 'b-content__title-record').text.strip()
    record = record_unclean[8:]
    df_record = pd.DataFrame([record], columns = ['record'])

    #For fight table
    fight_columns = [column.text.strip() for column in fighter_soup.find_all('th')]
    fight_rows = fighter_soup.find_all('tr')

    fight_data = []
    fight_ids = []
    for row in fight_rows:
        cells = row.find_all('td')
        clean_cell = [cell.text.strip() for cell in cells]
        if len(clean_cell) < len(fight_columns):
            clean_cell += [None] * (len(fight_columns) - len(clean_cell))
        
        fight_data.append(clean_cell)

        link = row.get('data-link')
        fight_ids.append(id_from_url(link) if link else None)

    '''
    AI FIX for if then above
    '''
    
    df_fight_table = pd.DataFrame(fight_data, columns = fight_columns)
    df_fight_table.insert(0, 'fight_id', fight_ids)
    
    #For biostats table
    biostats_table_soup = fighter_soup.find('ul')
    biostats_columns = [column.text.strip() for column in biostats_table_soup.find_all('i', class_ = 'b-list__box-item-title b-list__box-item-title_type_width')]

    biostats_data = []
    for i in range(len(biostats_columns)):
        biostats_data.append(biostats_table_soup.find_all('i', class_ = 'b-list__box-item-title b-list__box-item-title_type_width')[i].next_sibling.strip())
    df_biostats_table = pd.DataFrame([biostats_data], columns = biostats_columns)


    #For career statistics table
    career_stat_soup = fighter_soup.find('div', class_="b-list__info-box-left")
    career_stat_columns = [column.text.strip() for column in career_stat_soup.find_all('i')]

    career_stat_data = []
    for i in range(len(career_stat_columns)):
        career_stat_data.append(career_stat_soup.find_all('i')[i].next_sibling.strip())
    df_career_stat = pd.DataFrame([career_stat_data], columns = career_stat_columns)

    df_fighter_stats = pd.concat([df_record, df_career_stat, df_biostats_table], axis = 1).drop(columns=["Career statistics:"])
    
    return df_fighter_stats, df_fight_table

def single_fight_scraper(html):

    soup = BeautifulSoup(html, "html.parser")
    fight_soup = soup.find('div', class_ = 'b-fight-details')


    #For winner/loser/draw info:
    ordered_winner_result = [result.text.strip() for result in fight_soup.find_all('i', class_ = 'b-fight-details__person-status')]
    ordered_fighter_result = [result.text.strip() for result in fight_soup.find_all('a', class_ = 'b-link b-fight-details__person-link')]

    for i in range(2):
        if ordered_winner_result[i] == 'D':
            winner = 'TIE'
            loser = 'TIE'
        elif ordered_winner_result[i] == 'W':
            winner = ordered_fighter_result[i]
        else:
            loser = ordered_fighter_result[i]


    #For fight_details
    fight_details_soup = soup.find('p', class_ = 'b-fight-details__text')
    fight_details_columns = [column.text.strip() for column in fight_details_soup.find_all('i', class_ = 'b-fight-details__label')]

    fight_details_data = []

    fight_details_first_soup = fight_details_soup.find('i', class_ = "b-fight-details__text-item_first")
    fight_details_data.append(fight_details_first_soup.find('i', style="font-style: normal").text.strip())
    labels = fight_details_soup.find_all('i', class_ = 'b-fight-details__label')

    for column in labels:
        fight_details_data.append(column.next_sibling.text.strip())

    fight_details_data.pop(1) #remove blank space
    fight_details_columns.pop(4) #remove referee, since irrelevant
    fight_details_data.pop(4) #removes referee since irrelevant
    df_fight_details = pd.DataFrame([fight_details_data], columns = fight_details_columns)


    #round_by_round table
    rbr_soup = fight_soup.find('tr', class_ = "b-fight-details__table-row") # round by round soup
    rbr_columns = [column.text.strip() for column in rbr_soup] #has '' empty strings
    rbr_columns = [column for column in rbr_columns if column != ''] #removes empty strings

    rbr_fight_data_soup = fight_soup.find_all('section', class_ = "b-fight-details__section js-fight-section")[1]
    rbr_fight_datacell_soup = [column.text.strip() for column in rbr_fight_data_soup.find_all('p', class_ = "b-fight-details__table-text")]

    fighter_one_data = rbr_fight_datacell_soup[::2]
    fighter_two_data = rbr_fight_datacell_soup[1::2]

    df_rbr_total = pd.DataFrame(([fighter_one_data, fighter_two_data]), columns = rbr_columns)

    #aggregate fight table
    #round by round table
    #aggregate significant strikes table
    #round by round sig strikes table

    sections = fight_soup.find_all('section', class_="b-fight-details__section js-fight-section")

    index = [1, 2, 4]
    for i in index:
        section = sections[i]

        rbr_soup = section.find('tr', class_="b-fight-details__table-row")
        rbr_columns = [column.get_text(strip=True) for column in rbr_soup.find_all(['th', 'td'])]

        rbr_data = sections[i].find_all('p', class_="b-fight-details__table-text")
        rbr_data = [column.text.strip() for column in rbr_data]

        fighter_one_data = rbr_data[::2]
        fighter_two_data = rbr_data[1::2]
    
        arr_one = np.array(fighter_one_data)
        arr1 = arr_one.reshape(len(arr_one) // len(rbr_columns), len(rbr_columns))
        df_fighter_one = pd.DataFrame(arr1, columns=rbr_columns)

        arr_two = np.array(fighter_two_data)
        arr2 = arr_two.reshape(len(arr_two) // len(rbr_columns), len(rbr_columns))
        df_fighter_two = pd.DataFrame(arr2, columns=rbr_columns)

        if i == 1:
            df_aggregate_fight_1 = df_fighter_one
            df_aggregate_fight_2 = df_fighter_two
        if i == 2:
            df_rbr_fight_1 = df_fighter_one
            df_rbr_fight_2 = df_fighter_two
        if i == 4:
            df_sig_strikes_rbr_1 = df_fighter_one
            df_sig_strikes_rbr_2 = df_fighter_two


    #SCHEMA for TABLE

    df_rbr_fight_1.columns = ['Fighter','KD','Sig. str.','Sig. str. %','Total str.','Td','Td %','Sub. att','Rev.','Ctrl'] #reset scraping column names, scraped an error
    df_rbr_fight_2.columns = ['Fighter','KD','Sig. str.','Sig. str. %','Total str.','Td','Td %','Sub. att','Rev.','Ctrl'] #reset scraping column names, scraped an error

    df_aggregate_fight_1["Round"] = 0 #initialized round number
    df_aggregate_fight_2["Round"] = 0

    df_rbr_fight_1['Round'] = df_rbr_fight_1.index + 1
    df_rbr_fight_2['Round'] = df_rbr_fight_2.index + 1


    df_rounds_nonstrikes = pd.concat([df_aggregate_fight_1, df_rbr_fight_1, df_aggregate_fight_2, df_rbr_fight_2], axis = 0) #made composite table

    df_rbr_total['Round'] = 0
    df_sig_strikes_rbr_1['Round'] = df_sig_strikes_rbr_1.index + 1
    df_sig_strikes_rbr_2['Round'] = df_sig_strikes_rbr_2.index + 1

    df_rounds_strikes = pd.concat([df_rbr_total, df_sig_strikes_rbr_1, df_sig_strikes_rbr_2], axis = 0)
    df_rbr_fight  = pd.merge(df_rounds_nonstrikes, df_rounds_strikes, on = ['Fighter', 'Round'])


    df_winner = pd.DataFrame({"winner": [winner], 'loser': [loser]})
    df_fight_stats = pd.concat([df_fight_details, df_winner], axis = 1)

    return  df_fight_stats, df_rbr_fight
  
def fighter_url_scraper(subpage_html):

    soup = BeautifulSoup(subpage_html, "html.parser")

    row_soup = soup.find_all('tr', class_ = "b-statistics__table-row")
    fighter_urls = []

    for row in row_soup:
        a_tag = row.find('a')
        if a_tag:
            link = a_tag['href']
            fighter_urls.append(link)

    return fighter_urls

def single_raw_html_fetcher(url, page):
    page.goto(url, wait_until="domcontentloaded")
    try:
        page.wait_for_selector("body .l-page, table, tbody", timeout=15000)
    except:
        pass
    page.wait_for_timeout(200)
    html = page.content()
    return html

def events_page_scraper(page):
    url = 'http://ufcstats.com/statistics/events/completed?page=all'
    html = single_raw_html_fetcher(url, page)
    soup = BeautifulSoup(html, "html.parser")

    soup = soup.find('table', class_ = "b-statistics__table-events")
    row_soup = soup.find_all('a', class_ = 'b-link b-link_style_black')

    event_urls = []
    for row in row_soup:
        link = row.get('href')
        if link:
            event_urls.append(link)
    
    return event_urls

def fight_url_scraper(event_urls_list, page):

    fight_urls = []
    for url in event_urls_list:
        html = single_raw_html_fetcher(url, page)
        soup = BeautifulSoup(html, "html.parser")


        event_soup = soup.find('tbody', class_ = "b-fight-details__table-body")
        row_soup = event_soup.find_all('tr')

        for row in row_soup:
            link = row.get('data-link')
            if link:
                fight_urls.append(link)

    fight_urls = list(set(fight_urls)) # precautionary dedupe after iteration...
    return fight_urls

def save_progress(df, filename):
    df.to_csv( filename, mode="a", header=not os.path.exists(filename), index=False )

def id_from_url(url):
    return url.rstrip('/').split('/')[-1]

#

In [ ]:
#Data Pipeline

def run_pipeline():
    p = sync_playwright().start()
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    #Fighters pipeline
    subpage_urls_list = fighters_subpage_url_scraper()
    all_fighter_urls = []

    for i in range(len(subpage_urls_list)):
        all_fighter_urls.extend(fighter_url_scraper(single_raw_html_fetcher(subpage_urls_list[i], page)))

    pd.DataFrame(all_fighter_urls, columns=['fighter_url']).to_csv("all_fighter_urls.csv", index=False)

    for url in all_fighter_urls:
        try:
            df1, df2 = single_fighter_scraper(single_raw_html_fetcher(url, page))
            fid = id_from_url(url)
            df1.insert(0, 'fighter_id', fid)
            df2.insert(0, 'fighter_id', fid)
            save_progress(df1, "fighter_stats.csv")
            save_progress(df2, "fighter_fights.csv")
        except Exception as e:
            print(f"skip fighter {url}: {e}")
            df_fail = pd.DataFrame([[url, str(e)]], columns=['url', 'error'])
            save_progress(df_fail, "failed_fighters.csv")

    #Fights pipeline
    events_urls = events_page_scraper(page)
    complete_fight_urls = fight_url_scraper(events_urls, page)

    pd.DataFrame(events_urls, columns=['event_url']).to_csv("all_events_urls.csv", index=False)

    for url in complete_fight_urls:
        try:
            df3, df4 = single_fight_scraper(single_raw_html_fetcher(url, page))
            fid = id_from_url(url)
            df3.insert(0, 'fight_id', fid)
            df4.insert(0, 'fight_id', fid)
            save_progress(df3, "fight_oneline_stats.csv")
            save_progress(df4, "fights_roundbyround.csv")
        except Exception as e:
            print(f"skip fight {url}: {e}")
            df_fail = pd.DataFrame([[url, str(e)]], columns=['url', 'error'])
            save_progress(df_fail, "failed_fights.csv")


    page.close()
    browser.close()
    p.stop()

t = threading.Thread(target=run_pipeline)
t.start(); t.join()

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [8]:
import threading

def run_test():
    p = sync_playwright().start()
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    # --- one fighter ---
    furl = 'http://ufcstats.com/fighter-details/07f72a2a7591b409'
    try:
        df1, df2 = single_fighter_scraper(single_raw_html_fetcher(furl, page))
        fid = id_from_url(furl)
        df1.insert(0, 'fighter_id', fid)
        df2.insert(0, 'fighter_id', fid)
        save_progress(df1, "test_fighter_stats.csv")
        save_progress(df2, "test_fighter_fights.csv")
        print(f"fighter {fid}: stats {df1.shape}, fights {df2.shape}")
    except Exception as e:
        print(f"fighter FAIL: {e}")

    # --- one fight ---
    events_urls = events_page_scraper(page)
    fturl = fight_url_scraper(events_urls[:1], page)[0]
    try:
        df3, df4 = single_fight_scraper(single_raw_html_fetcher(fturl, page))
        ftid = id_from_url(fturl)
        df3.insert(0, 'fight_id', ftid)
        df4.insert(0, 'fight_id', ftid)
        save_progress(df3, "test_fight_stats.csv")
        save_progress(df4, "test_fights.csv")
        print(f"fight {ftid}: stats {df3.shape}, rbr {df4.shape}")
    except Exception as e:
        print(f"fight FAIL: {e}")

    page.close(); browser.close(); p.stop()

t = threading.Thread(target=run_test)
t.start(); t.join()

fighter 07f72a2a7591b409: stats (1, 16), fights (26, 12)
fight f1cb2e7ef096ec7c: stats (1, 7), rbr (8, 28)
